# Capstone ? Refresh opportunity scoring for search content

## FlyRank ML internship

This notebook turns the repo?s anonymized search intelligence data into a decision-support paper for refresh prioritisation. The model is trained to identify content pages likely to benefit from a refresh, while the narrative remains honest about what the model can and cannot claim.

### Abstract

Maintaining high-quality content is essential for sustained search visibility and user engagement. Using the anonymized FlyRank internship dataset, this capstone studies which content pages should be prioritized for refresh based on historical search performance, engagement, freshness, and content quality signals. A transparent baseline rule and a client-aware machine learning model are compared on the same validation split. On this sample, the learned model materially outperforms the baseline at identifying refresh candidates; the gain is strongest in the top-ranked queue, where the model is most relevant for editorial review. The final output is a ranked reviewer queue with interpretable reasons, designed to support human decision-making rather than replace it.

## 1. Question

The decision this supports is straightforward: which content pages should a reviewer inspect first for refresh, given limited editorial time and a need to improve discoverability without guessing?

The practical question is: among the thousands of content pages in the dataset, which pages are most likely to be declining, visible, and worth an update?

This is a prioritization task, not a causal claim about search-engine ranking. The model is treated as a decision-support tool for review queueing.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_all.py").exists()), Path.cwd())
feature_path = root / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.exists():
    subprocess.run([sys.executable, str(root / "scripts" / "01_prepare_features.py")], cwd=root, check=True)

df = pd.read_csv(feature_path)
print(f"Rows: {len(df):,}")
print(f"Positive label rate: {df['is_declining_label'].mean():.3f}")
print(df[["content_id", "client_id", "content_type", "avg_position", "ctr", "trend_direction", "is_declining_label"]].head().to_string(index=False))


Rows: 30,000
Positive label rate: 0.542
          content_id         client_id    content_type  avg_position  ctr trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc keyword article          10.6 0.76            down                   1
content_a1fb4e703a9e client_4e07408562 keyword article          20.3 0.05            down                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article          36.5 0.09            down                   1
content_331d6c4de07b client_19581e27de keyword article           6.2 0.49          stable                   0
content_d99b7a2d90ca client_3fdba35f04 keyword article          44.0 0.13            down                   1


## 2. Data

The data is the anonymized starter slice shipped with the internship repo: `data/raw/content_refresh_anonymized.csv`. It contains roughly 30,000 content rows across multiple clients, with search volume, engagement, age, freshness, and position signals. The target is `is_declining_label`, a public-safe label describing whether a page is in a declining refresh state based on its recent trend.

Public-safety rules are respected: no client names, page URLs, domains, titles, or private keywords are used. The queueing model is intentionally based on aggregate, anonymized indicators rather than anything that would reveal a client?s private strategy.

The repo?s data guide also flags several important caveats: rate columns are x100 percentages, `avg_position = 0` means no data, and `trend_direction` / `trend_pct` are never valid model features because the label is derived from them.

In [2]:
from pathlib import Path
import json
import pandas as pd

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_all.py").exists()), Path.cwd())
summary_path = root / "outputs" / "summary.json"
if not summary_path.exists():
    import subprocess, sys
    subprocess.run([sys.executable, str(root / "scripts" / "run_all.py")], cwd=root, check=True)

df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")
summary = json.loads(summary_path.read_text())
print(f"Rows scored: {summary['rows_scored']:,}")
print(f"Target positive rate: {summary['target_positive_rate']:.3f}")
print(f"Best model: {summary['best_model']}")
print("Missing values by column (top 8):")
print(df.isna().sum().sort_values(ascending=False).head(8).to_string())


Rows scored: 30,000
Target positive rate: 0.542
Best model: random_forest
Missing values by column (top 8):
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610


## 3. Methodology

The task is framed as a binary classification problem: given historical search and engagement signals, predict whether a content page should be surfaced for review as a likely refresh candidate. The validation design uses a client-aware holdout split rather than a random row split, which helps avoid leaking patterns from the same client into both training and evaluation.

The feature set combines engagement, visibility, freshness, content quality, and position signals. The baseline is a transparent rule-based queue built from search demand, page age, content depth, and position. The learned model compares logistic regression, decision tree, and random forest models using the same validation setup. Model selection is based on Precision@50, which is appropriate for a prioritization setting: the useful part of the queue is at the top, where a human reviewer has limited time.

Leakage was explicitly checked by avoiding target-derived columns and by respecting the client split. The label is not allowed to use post-hoc trend features that are directly derived from the target.

In [3]:
import json
from pathlib import Path
import pandas as pd

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'run_all.py').exists()), Path.cwd())
model_results = json.loads((root / 'outputs' / 'model_results.json').read_text())
summary = json.loads((root / 'outputs' / 'summary.json').read_text())
print('Split strategy:', model_results['split_strategy'])
print('Feature count:', model_results['feature_count'])
print('Target positive rate:', round(model_results['target_positive_rate'], 3))
print('Best model selected by Precision@50:', summary['best_model'])
print('Top feature importance:')
for item in model_results['best_model']['feature_importance_top'][:6]:
    print(f" - {item['feature']}: {item['importance']:.4f}")


Split strategy: client_holdout
Feature count: 52
Target positive rate: 0.542
Best model selected by Precision@50: random_forest
Top feature importance:
 - days_with_impressions: 0.1581
 - log_impressions_90d: 0.1286
 - avg_position: 0.1092
 - content_age_days: 0.0952
 - char_count: 0.0426
 - word_count: 0.0396


## 4. Results (vs baseline)

The model improves the top-ranked queue materially. The relevant comparison is not raw accuracy; the relevant metric is how well the model surfaces the most important review candidates early. That is why Precision@50 is the lead metric here.

The observed result is a strong lift over the transparent hand-rule baseline, with the random forest model winning on the client-aware evaluation split. The queue is then passed to a reviewer-facing action layer that maps the reasons to recommended actions such as `refresh`, `refresh_and_review_ctr`, and `refresh_and_review_engagement`.

In [4]:
import json
import pandas as pd
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'run_all.py').exists()), Path.cwd())
results = json.loads((root / 'outputs' / 'model_results.json').read_text())
baseline = results['baseline']
model_table = []
for name, metrics in results['models'].items():
    model_table.append({
        'Model': name,
        'ROC AUC': round(metrics['roc_auc'], 3),
        'Avg precision': round(metrics['average_precision'], 3),
        'Precision@50': round(metrics['precision_at_50'], 3),
        'Recall': round(metrics['recall'], 3),
        'F1': round(metrics['f1'], 3),
    })
model_table.append({
    'Model': 'baseline_rules',
    'ROC AUC': round(baseline['baseline_roc_auc'], 3),
    'Avg precision': round(baseline['baseline_average_precision'], 3),
    'Precision@50': round(baseline['baseline_precision_at_50'], 3),
    'Recall': '-',
    'F1': '-',
})
pd.DataFrame(model_table).sort_values('Precision@50', ascending=False).reset_index(drop=True)
print(pd.DataFrame(model_table).to_string(index=False))


              Model  ROC AUC  Avg precision  Precision@50 Recall     F1
      decision_tree    0.742          0.575          0.62  0.716  0.634
logistic_regression    0.700          0.522          0.40  0.567  0.566
      random_forest    0.750          0.618          0.74  0.744   0.64
     baseline_rules    0.627          0.468          0.24      -      -


## 5. Limitations

This work is a decision-support model built from a single anonymized sample and should not be interpreted as a universal statement about search engines, content quality, or causal effects. It describes measured patterns in the observed data, not an underlying law of the web.

The queue is strongest for prioritisation and weakest for causal inference. A page flagged by the model is not automatically a page that should be changed; editorial context, content strategy, and business priorities still matter. The holdout split reduces leakage risk, but it does not eliminate all drift risk across content types, traffic levels, or client dynamics.

The page queue also reflects the data?s real constraints: some columns are rate-encoded, some rows have no position data, and some content categories behave differently. The safe interpretation is directional and operational, not definitive.

In [5]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_all.py").exists()), Path.cwd())
feature_path = root / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.exists():
    subprocess.run([sys.executable, str(root / "scripts" / "01_prepare_features.py")], cwd=root, check=True)

df = pd.read_csv(feature_path)
print("No-data avg_position rows:", int((df["avg_position"] == 0).sum()))
print("Declining-label share:", round(df["is_declining_label"].mean(), 3))
print("Top rate-encoded columns to interpret carefully:")
for col in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct"]:
    print(f" - {col}: median={df[col].median():.3f}, max={df[col].max():.3f}")


No-data avg_position rows: 1205
Declining-label share: 0.542
Top rate-encoded columns to interpret carefully:
 - ctr: median=0.070, max=100.000
 - engagement_rate: median=0.000, max=100.000
 - scroll_rate: median=4.920, max=300.000
 - ai_traffic_pct: median=0.000, max=300.000
 - trend_pct: median=-26.000, max=44900.000


## 6. Ranked recommendations

The action playbook converts the score into a reviewer plan. The most common recommended actions are to refresh pages with low CTR and visible demand, improve engagement on high-visibility pages, or monitor pages that are not yet sufficiently risky to justify an immediate change.

In production, the expected workflow is: rank, inspect the top 20?50 pages, validate against editorial context, and then route a small, explainable subset to revision. This keeps the model useful without treating it as an autonomous publishing policy tool.

In [6]:
import pandas as pd
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_all.py").exists()), Path.cwd())
queue = pd.read_csv(root / "outputs" / "refresh_queue.csv")
top = queue[["final_rank", "final_refresh_score", "confidence", "suggested_action", "best_model_probability", "final_reason_codes", "impressions_90d", "sessions_90d", "trend_direction"]].head(10)
print(top.to_string(index=False))
print("\nAction counts:")
print(queue["suggested_action"].value_counts().to_string())


 final_rank  final_refresh_score confidence       suggested_action  best_model_probability                                                                                                                                                   final_reason_codes  impressions_90d  sessions_90d trend_direction
          1            81.734212       high refresh_and_review_ctr                0.783472 declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate            12834            66            down
          2            81.603243     medium refresh_and_review_ctr                0.849842                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate             2498             9            down
          3            81.544618       high refresh_and_review_ctr                0.789490 

## 7. Reproducibility

This notebook is designed to reproduce the published findings from the repo?s reference pipeline. The data and the model outputs are checked in within the repo?s public-safe limits, and the code paths are all built from the same source files.

To re-run the pipeline from the repository root:

```bash
python scripts/run_all.py
```

This regenerates the labelled feature vector, baseline queue, learned model, final ranked queue, charts, and the Markdown/PDF reports.

## 8. Acknowledgments & data credit

This project uses the anonymized FlyRank internship data described in the repo documentation and the public-safe data-use guidance. The data is intentionally anonymized and used for learning, decision support, and portfolio work.

Data credit: FlyRank internship data; see the repo docs and the project landing page at https://flyrank.ai.

## 9. Artifacts the paper embeds

The final paper embeds the charts and queue outputs that support the argument. These artifacts are not decorative; they are the evidence trail that turns the model into a review tool.

In [7]:
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_all.py").exists()), Path.cwd())
chart_dir = root / "outputs" / "charts"
for chart in sorted(chart_dir.glob('*.svg')):
    print(chart.name)


action_mix.svg
confidence_mix.svg
top_feature_importance.svg
top_reason_codes.svg
trend_distribution.svg


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled ? markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime ? Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] My paper has the required 9 sections, including Abstract and Acknowledgments & data credit
- [ ] ML-12 is included in the closing cells: 5-minute demo outline + social-post cut + a 3-sentence employer-facing summary


## ML-12 ? 5-minute demo outline

1. Hook: 'We had a choice between a hand rule and a model; the model gave us a much better top-of-queue review list.'
2. Data: 30k anonymized pages, client-aware split, public-safe metrics and no private data.
3. Method: baseline rules vs random forest, with Precision@50 as the operational metric.
4. Results: random forest increased top-ranked precision from 0.24 to 0.74, approximately a 3x lift in the review queue.
5. Decision: use the model to shortlist pages for review, then confirm with editorial judgment.


## ML-12 ? social-post cut

A transparent refresh model used on anonymized FlyRank data lifted top-of-queue precision from 0.24 to 0.74 on a client-aware validation split. That's the difference between a noisy backlog and a reviewer-ready shortlist.

#ML #DataScience #SEO #Search #Portfolio

## ML-12 ? employer-facing summary

I built a decision-support ranking model for content refresh prioritisation using anonymized search intelligence data. The project applied a client-aware validation split, compared a transparent rule baseline against learned models, and selected the best model on Precision@50 to optimize the top of the review queue. The results showed a strong lift in the most actionable review candidates, and the final output was an interpretable, reviewer-friendly queue for editorial decisions.